# Antarctica AT_crust_ice — Hybrid Wavefield & Receiver Functions

Run the **Python hybrid FDFK** engine on the `AT_crust_ice` Antarctica benchmark
(ice + crustal basement) and produce publication-style figures:

1. S-wave velocity model cross-section (0–5 km depth)
2. Combined wavefield snapshot figure (`uz` top row, `ux` bottom row)
3. **`uz` wavefield animation GIF** (cached snapshots, RdBu_r colormap)
4. Combined receiver waveforms (5 stations: `uz` left, `ux` right)
5. Moveout gathers (`ux`, `uz`)
6. P-wave receiver-function section from **hybrid seismograms** (SAC / seispy style: red = positive, blue = negative; display window −1–8 s)
7. **1D-ID RF profile section** — per-receiver vertical velocity column → Kennett FK synthetics → seispy iterative deconvolution

**Model:** `examples/Antarctica_RF/cache/AT_crust_ice/input/` (copied from `benchmark/AT_crust_ice` with identical settings).

**Outputs:** all figures saved under `examples/Antarctica_RF/` (`wavefield_uz.gif`, `rf_section_1d_id.png`, etc.).

> Computational cost: one Numba fused run produces **both** seismograms and snapshots
> (~10–20 s for nt≈6667 on a laptop). Cached outputs are reused unless
> `FORCE_REGEN` / `FORCE_SNAPSHOT_REGEN` is set. Alternative: `python run_workflow.py`.

In [3]:
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Project paths (notebook lives in examples/Antarctica_RF/)
EXAMPLE_DIR = Path(".").resolve()
PROJECT_ROOT = EXAMPLE_DIR.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from fk.io_fortran import load_example, read_su, write_su, read_fk_model
from fk.geometry import rayp_deg_to_si
from hybrid.config import HybridConfig
from hybrid.engine import compute_hybrid_seismograms
from rf import compute_prf
from plots.rf import sac_power_wiggle, rf_power_law, rf_amp_percentile

OUT_DIR = EXAMPLE_DIR
CACHE_DIR = OUT_DIR / "cache"
MODEL_DIR = CACHE_DIR / "AT_crust_ice" / "input"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

FORCE_REGEN = False          # set True to re-run the hybrid simulation
FORCE_SNAPSHOT_REGEN = False # set True to recompute wavefield snapshots
BWF_PAD_FACTOR = 1.0         # short 20 s record; anti-wraparound optional

print(f"Project root : {PROJECT_ROOT}")
print(f"Example dir  : {EXAMPLE_DIR}")
print(f"Model input  : {MODEL_DIR}")
print(f"Output dir   : {OUT_DIR}")

Project root : /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF
Model input  : /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/cache/AT_crust_ice/input
Output dir   : /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF


In [4]:
# --- Load AT_crust_ice case (same logic as benchmark/_common/run_python_benchmark.py) ---

def _model_stems(inpar_path: Path) -> tuple[str, str]:
    lines = [ln.strip() for ln in inpar_path.read_text().splitlines() if ln.strip()]
    return lines[5], lines[6]


def _without_pml_path(input_dir: Path, stem: str) -> Path:
    if stem.endswith(".su"):
        stem = stem[:-3]
    path = input_dir / f"{stem}_without_pml.su"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}; rebuild the AT_crust_ice model.")
    return path


input_dir = MODEL_DIR
ex = load_example(input_dir, side="left")
fd = ex.fd

nx = int(round((fd.xn - fd.x0) / fd.dx)) + 1
nz = int(round((fd.zn - fd.z0) / fd.dz)) + 1

config = HybridConfig(
    nx=nx, nz=nz,
    dx=fd.dx, dz=fd.dz,
    x0=fd.x0, z0=fd.z0,
    dt=fd.dt, nt=fd.nt,
    norder=fd.norder,
    npml=fd.npml,
)

vp_stem, vs_stem = _model_stems(input_dir / "inpar.dat")
vp_tr, _ = read_su(_without_pml_path(input_dir, vp_stem))
vs_tr, _ = read_su(_without_pml_path(input_dir, vs_stem))
vp_2d = vp_tr.T.astype(np.float64)
vs_2d = vs_tr.T.astype(np.float64)
rho_2d = (vp_2d + 980.0) / 2.760

model_left = read_fk_model(input_dir / "FK_model_left.dat")
model_right = read_fk_model(input_dir / "FK_model_right.dat")
p_si = rayp_deg_to_si(ex.src.rayp_deg)

vp_half = model_left.vp[model_left.halfspace_index]
sin_arg = min(1.0, max(-1.0, p_si * vp_half))
z_init = fd.zn + 0.5 * (nx - 1) * fd.dx * np.tan(np.arcsin(sin_arg))

SNAP_INTERVAL = fd.nstep  # snapshot every nstep steps (FD_model.dat: 100)

print(f"Grid: nx={nx}, nz={nz}, nt={fd.nt}, norder={config.norder}, npml={config.npml}")
print(f"dt={config.dt:.5f} s, dx={config.dx:.1f} m, dz={config.dz:.1f} m")
print(f"Source: {ex.source.incidence}-wave f0={ex.source.f0} Hz, rayp={ex.src.rayp_deg:.4f} s/deg")
print(f"Receivers: {ex.rx.size} (0–{ex.rx.max()/1000:.1f} km)")
print(f"z_init={z_init:.1f} m, snapshot interval={SNAP_INTERVAL} steps")

Grid: nx=301, nz=241, nt=6668, norder=3, npml=20
dt=0.00300 s, dx=50.0 m, dz=50.0 m
Source: P-wave f0=5.0 Hz, rayp=7.0197 s/deg
Receivers: 101 (0–10.0 km)
z_init=15251.1 m, snapshot interval=100 steps


In [5]:
# --- S-wave velocity model cross-section (0–5 km depth) ---

VS_DEPTH_MAX_KM = 5.0
extent_km = [config.x0 / 1000, config.xn / 1000, config.zn / 1000, config.z0 / 1000]
extent_vs_km = [extent_km[0], extent_km[1], VS_DEPTH_MAX_KM, extent_km[3]]

iz_max = min(vs_2d.shape[0], int(round(VS_DEPTH_MAX_KM * 1000 / fd.dz)) + 1)
vs_shallow = vs_2d[:iz_max, :]

from plots.wavefield_snapshots import annotate_interfaces, model_interface_profiles

interface_profiles_shallow = model_interface_profiles(
    vs_2d, config.x0, config.dx, fd.z0, fd.dz, max_depth_km=VS_DEPTH_MAX_KM,
)

with plt.rc_context({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
}):
    fig, ax = plt.subplots(figsize=(10, 3.6))
    im = ax.imshow(
        vs_shallow,
        aspect="auto",
        cmap="viridis",
        extent=extent_vs_km,
        interpolation="nearest",
        rasterized=True,
    )
    annotate_interfaces(
        ax, interface_profiles_shallow, color="white", lw=0.7, alpha=0.75,
    )
    ax.set_xlim(extent_vs_km[0], extent_vs_km[1])
    ax.set_ylim(extent_vs_km[2], extent_vs_km[3])
    ax.set_xlabel("Distance (km)")
    ax.set_ylabel("Depth (km)")
    ax.set_title("AT_crust_ice S-wave velocity model (0–5 km)", pad=8)
    ax.tick_params(labelsize=9, length=3, color="0.25")
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
        spine.set_color("0.35")
    cb = fig.colorbar(im, ax=ax, pad=0.012, fraction=0.035)
    cb.set_label("Vs (m/s)", fontsize=10)
    cb.ax.tick_params(labelsize=8)
    fig.tight_layout()
    vs_png = OUT_DIR / "vs_model_section.png"
    fig.savefig(vs_png, dpi=220, bbox_inches="tight")
    plt.close(fig)
print(f"Saved {vs_png}")


Saved /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/vs_model_section.png


In [6]:
# --- Run hybrid (one Numba pass: seismograms + snapshots when needed) ---

seisx_path = CACHE_DIR / "seisx.su"
seisz_path = CACHE_DIR / "seisz.su"
snap_cache = CACHE_DIR / "snapshots"
snap_marker = snap_cache / ".done"

need_seis = FORCE_REGEN or not (seisx_path.exists() and seisz_path.exists())
need_snaps = FORCE_SNAPSHOT_REGEN or FORCE_REGEN or not snap_marker.exists()

if need_seis or need_snaps:
    snap_iv = SNAP_INTERVAL if need_snaps else 0
    if snap_iv > 0:
        print(
            f"Running compute_hybrid_seismograms "
            f"(seismograms + snapshots, Numba fused loop, interval={snap_iv})..."
        )
    else:
        print("Running compute_hybrid_seismograms (Numba fused loop)...")
    t0 = time.perf_counter()
    result = compute_hybrid_seismograms(
        config,
        model_left,
        model_right,
        ex.source,
        p_si,
        receivers_x=ex.rx,
        receivers_z=ex.rz,
        vp_2d=vp_2d,
        vs_2d=vs_2d,
        rho_2d=rho_2d,
        phi=ex.phi,
        z_init=z_init,
        bwf_pad_factor=BWF_PAD_FACTOR,
        verbose=True,
        snapshot_interval=snap_iv,
    )
    print(f"Wall time: {time.perf_counter() - t0:.1f} s")

    if need_seis or snap_iv > 0:
        write_su(seisx_path, result.ux, config.dt)
        write_su(seisz_path, result.uz, config.dt)
    seisx, seisz = result.ux, result.uz
    t = result.t

    if need_snaps and snap_iv > 0:
        snap_cache.mkdir(parents=True, exist_ok=True)
        for i, (ux_snap, uz_snap) in enumerate(
            zip(result.snapshots_ux, result.snapshots_uz)
        ):
            it = i * SNAP_INTERVAL
            write_su(snap_cache / f"{it:05d}ux.su", ux_snap.T.astype(np.float32), config.dt)
            write_su(snap_cache / f"{it:05d}uz.su", uz_snap.T.astype(np.float32), config.dt)
        snap_marker.write_text("ok")
        snapshots_ux = result.snapshots_ux
        snapshots_uz = result.snapshots_uz
    elif snap_marker.exists():
        import glob
        print(f"Loading cached snapshots from {snap_cache}")
        ux_files = sorted(glob.glob(str(snap_cache / "*ux.su")))
        snapshots_ux, snapshots_uz = [], []
        for f in ux_files:
            stem = Path(f).name.replace("ux.su", "")
            ux_tr, _ = read_su(f)
            uz_tr, _ = read_su(snap_cache / f"{stem}uz.su")
            snapshots_ux.append(ux_tr.T)
            snapshots_uz.append(uz_tr.T)
    else:
        snapshots_ux, snapshots_uz = [], []
else:
    print(f"Loading cached seismograms from {CACHE_DIR}")
    seisx, dt_x = read_su(seisx_path)
    seisz, dt_z = read_su(seisz_path)
    assert abs(dt_x - config.dt) < 1e-9
    t = np.arange(seisx.shape[1]) * config.dt

    import glob
    print(f"Loading cached snapshots from {snap_cache}")
    ux_files = sorted(glob.glob(str(snap_cache / "*ux.su")))
    snapshots_ux, snapshots_uz = [], []
    for f in ux_files:
        stem = Path(f).name.replace("ux.su", "")
        ux_tr, _ = read_su(f)
        uz_tr, _ = read_su(snap_cache / f"{stem}uz.su")
        snapshots_ux.append(ux_tr.T)
        snapshots_uz.append(uz_tr.T)

nrcv, nt = seisx.shape
print(f"Seismograms: {nrcv} traces × {nt} samples, dt={config.dt:.5f} s")
print(f"Snapshots available: {len(snapshots_uz)}")

Running compute_hybrid_seismograms (no snapshots)...
[Hybrid] Computing FK background wavefield (nz=241, nx=301, nt=6668)...


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


[Hybrid] Background wavefield memory: 423.0 MB
[Hybrid] Initializing FD solver...
[Hybrid] Time-stepping: 6668 steps, dt=0.00300s...
[Hybrid] Simulation complete.
Wall time: 8.3 s
Seismograms: 101 traces × 6668 samples, dt=0.00300 s


In [7]:
# Seismograms and snapshots are produced together in the cell above.
# Or run: python run_workflow.py

Running hybrid with snapshot_interval=100 (slow)...
[Hybrid] Computing FK background wavefield (nz=241, nx=301, nt=6668)...
[Hybrid] Background wavefield memory: 423.0 MB
[Hybrid] Initializing FD solver...
[Hybrid] Time-stepping: 6668 steps, dt=0.00300s...
[Hybrid] Simulation complete.
Snapshot run wall time: 19.3 s
Snapshots available: 67


In [8]:
# --- Plot wavefield snapshot panels (publication style; see 2.2_hybrid_vs_fortran_Altyn.ipynb) ---

from matplotlib.colors import Normalize
from plots.wavefield_snapshots import annotate_interfaces, model_interface_profiles

SNAPSHOT_MAX_COLUMNS = 5
SNAPSHOT_PERCENTILE = 99.5
extent_km = [config.x0 / 1000, config.xn / 1000, config.zn / 1000, config.z0 / 1000]
# Skip iz=0 in snapshot panels: that row is a free-surface ghost node set by
# stress-free extrapolation (_apply_free_surface), not by the FD update. Plotting
# it at z=0 shows a spurious high-frequency stripe (worst in ux). Receivers and
# the physical wavefield use iz>=1; shallow reverberations below are unchanged.
SNAPSHOT_IZ_PLOT_START = 1
extent_plot_km = [
    extent_km[0], extent_km[1], extent_km[2],
    (config.z0 + config.dz) / 1000,
]


def snapshot_for_plot(data: np.ndarray) -> np.ndarray:
    """Return snapshot rows for display (drops the surface ghost row)."""
    return data[SNAPSHOT_IZ_PLOT_START:, :]


def choose_snapshot_indices(n_snaps: int, max_cols: int = SNAPSHOT_MAX_COLUMNS) -> list[int]:
    """Select evenly spaced snapshot indices for a readable panel figure."""
    if n_snaps <= max_cols:
        return list(range(n_snaps))
    idx = np.linspace(0, n_snaps - 1, max_cols, dtype=int)
    return [int(i) for i in idx]


def symmetric_limit(arrays: list[np.ndarray], percentile: float = SNAPSHOT_PERCENTILE) -> float:
    """Robust symmetric color limit across a list of wavefield arrays."""
    values = np.concatenate([np.ravel(np.abs(a[np.isfinite(a)])) for a in arrays])
    if values.size == 0:
        return 1.0
    limit = float(np.percentile(values, percentile))
    return limit if limit > 1e-30 else 1.0


def style_snapshot_axis(ax, *, show_xlabel: bool, show_ylabel: bool) -> None:
    ax.set_xlim(extent_plot_km[0], extent_plot_km[1])
    ax.set_ylim(extent_plot_km[2], extent_plot_km[3])
    ax.tick_params(labelsize=9, length=3, color="0.25")
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
        spine.set_color("0.35")
    if show_xlabel:
        ax.set_xlabel("Distance (km)", fontsize=10)
    else:
        ax.set_xlabel("")
        ax.set_xticklabels([])
    if show_ylabel:
        ax.set_ylabel("Depth (km)", fontsize=10)
    else:
        ax.set_ylabel("")
        ax.set_yticklabels([])


def plot_snapshots(snapshots_uz, snapshots_ux, out_png: Path) -> None:
    """Plot uz (top row) and ux (bottom row) snapshots in one publication-style figure."""
    idxs = choose_snapshot_indices(len(snapshots_uz))
    ncols = len(idxs)
    nrows = 2
    row_labels = ("uz", "ux")
    components = (snapshots_uz, snapshots_ux)

    wave_limit = symmetric_limit(
        [snapshot_for_plot(snapshots_uz[i]) for i in idxs]
        + [snapshot_for_plot(snapshots_ux[i]) for i in idxs],
        SNAPSHOT_PERCENTILE,
    )
    interface_profiles = model_interface_profiles(vs_2d, config.x0, config.dx, fd.z0, fd.dz)
    wave_norm = Normalize(vmin=-wave_limit, vmax=wave_limit)

    fig_width = max(11.0, 2.65 * ncols + 1.4)
    fig_height = 2.15 * nrows + 1.1
    with plt.rc_context({
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
    }):
        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(fig_width, fig_height),
            sharex=True,
            sharey=True,
            squeeze=False,
            constrained_layout=False,
        )
        fig.subplots_adjust(left=0.08, right=0.86, top=0.84, bottom=0.12,
                            wspace=0.04, hspace=0.08)

        wave_im = None
        for col, si in enumerate(idxs):
            t_sec = si * SNAP_INTERVAL * config.dt
            axes[0, col].set_title(f"t = {t_sec:.1f} s", pad=8)

            for row, comp_snapshots in enumerate(components):
                data = snapshot_for_plot(comp_snapshots[si])
                im = axes[row, col].imshow(
                    data,
                    aspect="auto",
                    cmap="RdBu_r",
                    norm=wave_norm,
                    extent=extent_plot_km,
                    interpolation="nearest",
                    rasterized=True,
                )
                if wave_im is None:
                    wave_im = im
                annotate_interfaces(axes[row, col], interface_profiles)
                style_snapshot_axis(
                    axes[row, col],
                    show_xlabel=(row == nrows - 1),
                    show_ylabel=(col == 0),
                )
                if col == 0:
                    axes[row, col].text(
                        -0.16, 0.5, row_labels[row], transform=axes[row, col].transAxes,
                        rotation=90, va="center", ha="center", fontsize=11,
                        fontweight="semibold",
                    )

        fig.suptitle("AT_crust_ice wavefield snapshots", fontsize=14, fontweight="semibold", y=1.0)

        if wave_im is not None:
            top_pos = axes[0, -1].get_position()
            bot_pos = axes[1, -1].get_position()
            cax_wave = fig.add_axes([
                top_pos.x1 + 0.012, bot_pos.y0, 0.015, top_pos.y1 - bot_pos.y0,
            ])
            cb = fig.colorbar(wave_im, cax=cax_wave)
            cb.set_label("displacement (m)", fontsize=10)
            cb.ax.tick_params(labelsize=8)

        fig.savefig(out_png, dpi=220, bbox_inches="tight")
        # plt.close(fig)
    print(f"Saved {out_png}")


plot_snapshots(snapshots_uz, snapshots_ux, OUT_DIR / "wavefield_snapshots.png")


Saved /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/wavefield_snapshots.png


In [9]:
# --- Animate uz wavefield snapshots (GIF) ---

from plots.animate_wavefield import animate_wavefield

GIF_FPS = 15
GIF_DPI = 100

# Match symmetric color limits from the static snapshot figure
_gif_limit = symmetric_limit(snapshots_uz, SNAPSHOT_PERCENTILE)
gif_path = OUT_DIR / "wavefield_uz.gif"

animate_wavefield(
    snap_cache,
    comp="uz",
    fd_model_path=MODEL_DIR / "FD_model.dat",
    out_path=gif_path,
    fps=GIF_FPS,
    dpi=GIF_DPI,
    vmin=-_gif_limit,
    vmax=_gif_limit,
    cmap="RdBu_r",
)
print(f"Saved {gif_path}")

Found 67 uz snapshot files in /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/cache/snapshots
  Grid: x=[0, 15000] m, z=[0, 12000] m, dx=50 m, dz=50 m
  Snapshot shape: nx=301, nz=241
Saved animation to /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/wavefield_uz.gif
Saved /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/wavefield_uz.gif


In [10]:
# --- Receiver waveforms at five stations (uz left, ux right) ---

def overlay_trace(tr: np.ndarray) -> np.ndarray:
    """Peak-normalize for waveform comparison."""
    return tr / (np.max(np.abs(tr)) + 1e-30)


N_RCV_PLOT = 5
rcv_plot_idx = np.linspace(0, nrcv - 1, N_RCV_PLOT, dtype=int)

fig, axes = plt.subplots(len(rcv_plot_idx), 2, figsize=(14, 3 * len(rcv_plot_idx)),
                         sharex=True)

for row, idx in enumerate(rcv_plot_idx):
    x_km = ex.rx[idx] / 1000

    ax = axes[row, 0]
    ax.plot(t, overlay_trace(seisz[idx]), "k-", lw=0.8)
    ax.set_ylabel(f"x = {x_km:.1f} km")
    if row == 0:
        ax.set_title("uz (vertical)")

    ax = axes[row, 1]
    ax.plot(t, overlay_trace(seisx[idx]), "k-", lw=0.8)
    if row == 0:
        ax.set_title("ux (horizontal)")

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
fig.suptitle("AT_crust_ice receiver waveforms (peak-normalized)", y=1.01)
fig.tight_layout()
out_png = OUT_DIR / "receiver_waveforms.png"
fig.savefig(out_png, dpi=150)
# plt.close(fig)
print(f"Saved {out_png}")

Saved /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/receiver_waveforms.png


In [11]:
# --- Moveout gathers (distance vs time, wiggle fill) ---

def plot_moveout(data: np.ndarray, comp: str, out_png: Path,
                 decimate: int = 2, scale: float = 0.6) -> None:
    """Wiggle moveout: y = receiver distance, x = time."""
    x_km = ex.rx / 1000.0
    indices = np.arange(0, nrcv, decimate)
    dx_km = (x_km[1] - x_km[0]) if nrcv > 1 else 0.1

    fig, ax = plt.subplots(figsize=(10, 7))
    for j, idx in enumerate(indices):
        tr = overlay_trace(data[idx])
        y0 = x_km[idx]
        y = y0 + tr * dx_km * scale
        ax.plot(t, y, "k", lw=0.5)
        ax.fill_between(t, y0, y, where=(y > y0), color="k", alpha=0.5, linewidth=0)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance along profile (km)")
    ax.set_title(f"AT_crust_ice {comp} moveout (peak-normalized wiggles)")
    ax.set_ylim(x_km[0] - dx_km, x_km[-1] + dx_km)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    # plt.close(fig)
    print(f"Saved {out_png}")


plot_moveout(seisz, "uz", OUT_DIR / "moveout_uz.png")
plot_moveout(seisx, "ux", OUT_DIR / "moveout_ux.png")

Saved /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/moveout_uz.png
Saved /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/moveout_ux.png


In [12]:
# --- Receiver functions (iterative deconvolution, benchmark RF settings) ---

RF_F0 = 8.0       # Gaussian width (plot_result.m / benchmark default)
RF_ITMAX = 100
RF_MINDERR = 0.001
RF_TSHIFT = 10.0  # seispy convention: seconds before direct P (deconvolution only)
T_RF_MIN = -1.0   # RF plot display window (s), relative to direct P
T_RF_MAX = 8.0

rf_npz = OUT_DIR / "receiver_functions.npz"
if FORCE_REGEN or not rf_npz.exists():
    print("Computing P-wave receiver functions...")
    rfs, t_rf, rms_arr = compute_prf(
        seisx, seisz, config.dt,
        tshift=RF_TSHIFT, f0=RF_F0,
        method="iter", itmax=RF_ITMAX, minderr=RF_MINDERR,
    )
    print(f"RF shape: {rfs.shape}, mean RMS: {rms_arr.mean():.4f}")
    np.savez(
        rf_npz,
        rfs=rfs, t_rf=t_rf, dt=config.dt, rms=rms_arr,
        f0=RF_F0, tshift=RF_TSHIFT,
    )
    print(f"Saved RF data to {rf_npz}")
else:
    print(f"Loading cached receiver functions from {rf_npz}")
    cached = np.load(rf_npz)
    rfs, t_rf, rms_arr = cached["rfs"], cached["t_rf"], cached["rms"]
    print(f"RF shape: {rfs.shape}, mean RMS: {rms_arr.mean():.4f}")

Computing P-wave receiver functions...
RF shape: (101, 6668), mean RMS: 0.9227
Saved RF data to /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/receiver_functions.npz


In [13]:
# --- SAC-style RF section (red positive / blue negative wiggles) ---

RF_POWER = 2.0
RF_AMP_PCT = 80.0
BIN_SIZE_KM = 0.08      # horizontal wiggle scale (km)
DECIMATE_RF = 1         # plot every trace

disp_mask = (t_rf >= T_RF_MIN) & (t_rf <= T_RF_MAX)
amp_mask = (t_rf >= 0.0) & (t_rf <= T_RF_MAX)  # post-P window for amplitude reference
rfs_pow = rf_power_law(rfs, RF_POWER)
amp_ref = rf_amp_percentile(rfs_pow, mask=amp_mask, percentile=RF_AMP_PCT)

x_km = ex.rx / 1000.0
indices = np.arange(0, nrcv, DECIMATE_RF)

t_plot = t_rf[disp_mask]

fig, ax = plt.subplots(figsize=(10, 8))
for j, idx in enumerate(indices):
    wiggle = sac_power_wiggle(rfs[idx], amp_ref, BIN_SIZE_KM, power=RF_POWER)[disp_mask]
    x_pos = x_km[idx]
    ax.plot(x_pos + wiggle, t_plot, "k", lw=0.4)
    ax.fill_betweenx(t_plot, x_pos, x_pos + wiggle,
                     where=(wiggle > 0), color="red", alpha=0.45, linewidth=0)
    ax.fill_betweenx(t_plot, x_pos, x_pos + wiggle,
                     where=(wiggle < 0), color="blue", alpha=0.45, linewidth=0)

ax.axhline(0.0, color="gray", ls="--", lw=0.8, label="P arrival")
ax.set_xlabel("Distance along profile (km)")
ax.set_ylabel("Time lag (s)")
ax.set_title(
    f"P-wave RF section — AT_crust_ice (f0={RF_F0}, iter, n={nrcv} traces)\n"
    "SAC style: red = positive, blue = negative"
)
ax.set_ylim(T_RF_MAX, T_RF_MIN)  # time increases downward (seismic convention)
ax.set_xlim(x_km[0] - BIN_SIZE_KM, x_km[-1] + BIN_SIZE_KM)
ax.grid(alpha=0.25)
fig.tight_layout()
rf_png = OUT_DIR / "rf_section_sac.png"
fig.savefig(rf_png, dpi=150)
# plt.close(fig)
print(f"Saved {rf_png}")

Saved /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/rf_section_sac.png


## 1D-ID receiver-function profile

For each surface receiver we build a **local 1D velocity model** by extracting the
`Vp`, `Vs`, and `ρ` grid column directly beneath the station. Constant-velocity
runs are merged into layers; layer tops define a `FKLayerModel`.

Synthetic **radial (`ux`) and vertical (`uz`) seismograms** are computed with the
project FK engine using the **Kennett reflectivity** propagator (`kennett_numba`,
the default Haskell–Kennett / “HK” 1D method). Source, ray parameter, `z_init`,
and `phi` match the hybrid run above.

P-wave receiver functions follow the same **seispy iterative deconvolution**
(`rf.compute_prf`, Ligorria & Ammon 1999) and SAC-style plotting as the hybrid
RF section (display window −1–8 s).

In [14]:
# --- 1D-ID RF: column FK synthetics + iterative deconvolution ---

from fk.model import FKLayerModel
from fk.engine import compute_fk_seismograms


def column_to_fk_model(ix: int, z0: float, dz: float) -> FKLayerModel:
    """Merge a 2D model column into a layered FK model at velocity jumps."""
    vp_col = vp_2d[:, ix]
    vs_col = vs_2d[:, ix]
    rho_col = rho_2d[:, ix]
    nz_col = vp_col.size
    z_depths = z0 + np.arange(nz_col) * dz
    changed = np.ones(nz_col, dtype=bool)
    changed[1:] = (np.abs(np.diff(vp_col)) > 1e-3) | (np.abs(np.diff(vs_col)) > 1e-3)
    layer_idx = np.where(changed)[0]
    return FKLayerModel.from_layers(
        vp_col[layer_idx], vs_col[layer_idx], z_depths[layer_idx], rho=rho_col[layer_idx]
    )


def receiver_column_index(x_m: float) -> int:
    """Map receiver x (m) to the nearest interior grid column."""
    ix = int(round((x_m - config.x0) / config.dx))
    return int(np.clip(ix, 0, vp_2d.shape[1] - 1))


rf_id_npz = OUT_DIR / "receiver_functions_1d_id.npz"
if FORCE_REGEN or not rf_id_npz.exists():
    print("Computing 1D-ID FK synthetics per receiver (Kennett reflectivity)...")
    seisx_id = np.zeros((nrcv, nt), dtype=np.float64)
    seisz_id = np.zeros((nrcv, nt), dtype=np.float64)
    t0_id = time.perf_counter()
    for i in range(nrcv):
        ix = receiver_column_index(ex.rx[i])
        model_id = column_to_fk_model(ix, fd.z0, fd.dz)
        fk_res = compute_fk_seismograms(
            model_id,
            ex.source,
            [ex.rx[i]],
            config.dt,
            nt,
            p_si=p_si,
            receivers_z=[ex.rz[i]],
            x0_plane=ex.x0_plane,
            phi=ex.phi,
            model_max_depth=fd.zn,
            z_init=z_init,
            method="kennett_numba",
        )
        seisx_id[i] = fk_res.ux[0]
        seisz_id[i] = fk_res.uz[0]
    print(f"FK synthetics wall time: {time.perf_counter() - t0_id:.1f} s")

    print("Iterative RF deconvolution (seispy via rf.compute_prf)...")
    rfs_id, t_rf_id, rms_id = compute_prf(
        seisx_id,
        seisz_id,
        config.dt,
        tshift=RF_TSHIFT,
        f0=RF_F0,
        method="iter",
        itmax=RF_ITMAX,
        minderr=RF_MINDERR,
    )
    np.savez(
        rf_id_npz,
        rfs=rfs_id,
        t_rf=t_rf_id,
        seisx=seisx_id,
        seisz=seisz_id,
        rms=rms_id,
        dt=config.dt,
    )
    print(f"Saved 1D-ID RF data to {rf_id_npz}")
else:
    print(f"Loading cached 1D-ID receiver functions from {rf_id_npz}")
    cached_id = np.load(rf_id_npz)
    rfs_id, t_rf_id, rms_id = cached_id["rfs"], cached_id["t_rf"], cached_id["rms"]
    print(f"RF shape: {rfs_id.shape}, mean RMS: {rms_id.mean():.4f}")

Computing 1D-ID FK synthetics per receiver (Kennett reflectivity)...
FK synthetics wall time: 1.3 s
Iterative RF deconvolution (seispy via rf.compute_prf)...
Saved 1D-ID RF data to /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/receiver_functions_1d_id.npz


In [15]:
# --- SAC-style 1D-ID RF section ---

disp_mask_id = (t_rf_id >= T_RF_MIN) & (t_rf_id <= T_RF_MAX)
amp_mask_id = (t_rf_id >= 0.0) & (t_rf_id <= T_RF_MAX)
rfs_id_pow = rf_power_law(rfs_id, RF_POWER)
amp_ref_id = rf_amp_percentile(rfs_id_pow, mask=amp_mask_id, percentile=RF_AMP_PCT)

t_plot_id = t_rf_id[disp_mask_id]

fig, ax = plt.subplots(figsize=(10, 8))
for idx in range(nrcv):
    wiggle = sac_power_wiggle(rfs_id[idx], amp_ref_id, BIN_SIZE_KM, power=RF_POWER)[disp_mask_id]
    x_pos = x_km[idx]
    ax.plot(x_pos + wiggle, t_plot_id, "k", lw=0.4)
    ax.fill_betweenx(
        t_plot_id, x_pos, x_pos + wiggle,
        where=(wiggle > 0), color="red", alpha=0.45, linewidth=0,
    )
    ax.fill_betweenx(
        t_plot_id, x_pos, x_pos + wiggle,
        where=(wiggle < 0), color="blue", alpha=0.45, linewidth=0,
    )

ax.axhline(0.0, color="gray", ls="--", lw=0.8, label="P arrival")
ax.set_xlabel("Distance along profile (km)")
ax.set_ylabel("Time lag (s)")
ax.set_title(
    f"1D-ID P-wave RF section — column FK + iterative decon (f0={RF_F0}, n={nrcv})\n"
    "SAC style: red = positive, blue = negative"
)
ax.set_ylim(T_RF_MAX, T_RF_MIN)
ax.set_xlim(x_km[0] - BIN_SIZE_KM, x_km[-1] + BIN_SIZE_KM)
ax.grid(alpha=0.25)
fig.tight_layout()
rf_id_png = OUT_DIR / "rf_section_1d_id.png"
fig.savefig(rf_id_png, dpi=150)
plt.close(fig)
print(f"Saved {rf_id_png}")

Saved /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/rf_section_1d_id.png


In [16]:
# --- Single-station RF example (middle receiver) ---

mid = nrcv // 2
rf = rfs[mid]
t_plot = t_rf[disp_mask]
rf_plot = rf[disp_mask]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_plot, rf_plot, "k", lw=1.0)
ax.fill_between(t_plot, rf_plot, where=(rf_plot > 0), color="red", alpha=0.4)
ax.fill_between(t_plot, rf_plot, where=(rf_plot < 0), color="blue", alpha=0.4)
ax.axvline(0.0, color="gray", ls="--", lw=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title(f"P-wave RF — trace {mid} (x = {ex.rx[mid]/1000:.1f} km)")
ax.set_xlim(T_RF_MIN, T_RF_MAX)
ax.grid(alpha=0.3)
fig.tight_layout()
single_png = OUT_DIR / "rf_single_mid.png"
fig.savefig(single_png, dpi=150)
# plt.close(fig)
print(f"Saved {single_png}")
print("\nAll figures written to:", OUT_DIR)
print("  Includes wavefield_uz.gif, rf_section_1d_id.png, receiver_functions_1d_id.npz")

Saved /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF/rf_single_mid.png

All figures written to: /Users/jygong/Stanford/Courses/GP245 Computational Earthsystem Analysis/Final Project/FDFK_Python/examples/Antarctica_RF
  Includes wavefield_uz.gif, rf_section_1d_id.png, receiver_functions_1d_id.npz
